# Week 5: choosing the alert threshold, top features, and a retraining demonstration

COMP6310 group project. The group decided (November 2026) that:

- the project uses **offline static data** with predefined features, not live data streams;
- an **alert** comes from the model's flag at a **threshold**, and the threshold is chosen **after testing**,
  using the top features to explain and justify it;
- **XGBoost** is the main model, with **one comparison model** (Random Forest, see
  `ddos_update4_experiments.ipynb`);
- retraining as behaviour changes is answered by **showing how XGBoost is retrained** with new labelled data.

This notebook provides the evidence for the last two points. Sections 0 to 6 and the split and model helpers
are copied unchanged from `ddos_update4_experiments.ipynb`, so the data and the splits are the same as the
desktop app's. Sections 9 to 11 are new:

- **9.** the app's trained models (`DDoSDetector/models/*_f9p`) scored at many thresholds;
- **10.** the top features (SHAP), including why missed attack flows score low;
- **11.** a retraining demonstration: a new attack tool appears, analysts label some of its flows, the model is
  retrained, and we measure what changes.

The model score is the model's raw output, not calibrated, so it is not a probability. Running the notebook
takes about 15 minutes.

## 0. Imports

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

## 1. Settings (as in v2, plus the destination port for feature 7)

In [2]:
DATA_DIR = Path("../data")      # put the CSV files here; do not commit them
FILES = ["Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
         "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"]

LABEL, TIME = "Label", "Timestamp"
PORT = "Dst Port"               # feature 7 only: used to count flows, never a model input
WINDOW_SECONDS = 10
BASE_FEATURES = [
    "Flow Pkts/s", "Flow Byts/s", "Flow Duration",
    "Flow IAT Mean", "Flow IAT Std", "SYN Flag Cnt",
    "Fwd Pkt Len Mean", "Fwd Pkt Len Std",
]
COUNT_COLS = ["Tot Fwd Pkts", "Tot Bwd Pkts"]   # used to build the fwd/bwd ratio feature
CANDIDATE_COLS = ["TotLen Fwd Pkts", "TotLen Bwd Pkts", "Fwd IAT Mean", "Fwd IAT Min",
                  "Active Mean", "Idle Mean", "PSH Flag Cnt", "Fwd Act Data Pkts"]
EXTRA_COLS = ["Bwd Pkt Len Mean", "RST Flag Cnt", "FIN Flag Cnt"] + CANDIDATE_COLS

SEEDS = [0, 1]
BENIGN_CAP = 1_000_000          # benign flows in each training set (as train_model.py)
TRAIN_FRAC = 0.7

## 2. Load the data (v2's loader; `Dst Port` is kept as text)

In [3]:
def load(path):
    head = pd.read_csv(path, nrows=0).columns.tolist()
    print(path.name, "-", len(head), "columns | has Dst Port:", PORT in head)
    for need in [LABEL, TIME, PORT] + BASE_FEATURES + COUNT_COLS:
        if need not in head:
            print("   MISSING column:", need)
    for opt in EXTRA_COLS:
        if opt not in head:
            print("   optional column missing (skipped):", opt)
    wanted = [LABEL, TIME, PORT] + BASE_FEATURES + COUNT_COLS + EXTRA_COLS
    keep = [c for c in head if c in set(wanted)]
    df = pd.read_csv(path, usecols=keep, low_memory=False, dtype={PORT: str})
    df = df[df[LABEL] != LABEL].copy()
    for c in BASE_FEATURES + COUNT_COLS + EXTRA_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if TIME in df:
        df[TIME] = pd.to_datetime(df[TIME], dayfirst=True, errors="coerce")
    df["source_file"] = path.name
    return df

t_start = time.time()
df = pd.concat([load(DATA_DIR / f) for f in FILES], ignore_index=True)
print(df.shape, f"({time.time() - t_start:.0f} s)")

Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv - 84 columns | has Dst Port: True


Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv - 80 columns | has Dst Port: True


(8997323, 25) (56 s)


## 3. Keep benign and DDoS flows, and name the tool (as v2)

In [4]:
def tool_of(label):
    s = str(label).lower()
    if "benign" in s: return "Benign"
    if "udp" in s:    return "LOIC-UDP"
    if "hoic" in s:   return "HOIC"
    if "http" in s:   return "LOIC-HTTP"
    return "Other"

df["tool"] = df[LABEL].map(tool_of)
df = df[df["tool"] != "Other"].copy()
df["y"] = (df["tool"] != "Benign").astype(int)     # 1 = DDoS (positive class)
print(df["tool"].value_counts())

tool
Benign       7733390
HOIC          686012
LOIC-HTTP     576191
LOIC-UDP        1730
Name: count, dtype: int64


## 4. Feature 7: flows to the same destination port in the previous 10 seconds (new)

For a flow that starts in second *s*, feature 7 counts the flows **of the same file** to the **same
destination port** that started in seconds *s*−10 to *s*−1. It looks back only (it could be counted while
traffic arrives) and does not count flows in the same second.

It is counted on **every row of each file, before cleaning**, exactly as the desktop app counts it for the
file it opens (`DDoSDetector/app/core/features.py`, `_port_window`). Times and ports are used only to
count; they are never model inputs. pandas 3 stores these times in microseconds, so seconds are computed
with `total_seconds()` rather than by dividing the raw integer.

In [5]:
def port_window(d):
    sec = np.floor((d[TIME] - pd.Timestamp("1970-01-01")).dt.total_seconds())
    key = pd.DataFrame({"cap": d["source_file"], "port": pd.to_numeric(d[PORT], errors="coerce"),
                        "sec": sec}).dropna()
    c = key.groupby(["cap", "port", "sec"]).size().rename("n").reset_index().sort_values(["cap", "port", "sec"])
    c["before"] = c.groupby(["cap", "port"])["n"].cumsum() - c["n"]             # flows with sec < s
    look = c[["cap", "port", "sec"]].assign(target=c["sec"] - WINDOW_SECONDS).sort_values("target")
    right = c[["cap", "port", "sec", "before", "n"]].rename(columns={"sec": "rsec"}).sort_values("rsec")
    m = pd.merge_asof(look, right, left_on="target", right_on="rsec", by=["cap", "port"],
                      direction="backward", allow_exact_matches=False)
    m["before_window"] = (m["before"] + m["n"]).fillna(0)                        # flows with sec < s-10
    c = c.merge(m[["cap", "port", "sec", "before_window"]], on=["cap", "port", "sec"], how="left")
    c["count"] = c["before"] - c["before_window"]
    rows = key.reset_index().merge(c[["cap", "port", "sec", "count"]], on=["cap", "port", "sec"], how="left")
    out = pd.Series(np.nan, index=d.index)
    out.loc[rows["index"].to_numpy()] = rows["count"].to_numpy(dtype="float64")
    return out

df["dst_port_flows_10s"] = port_window(df)                                       # feature 7

# Check against a brute-force count on 100 random rows
sec = np.floor((df[TIME] - pd.Timestamp("1970-01-01")).dt.total_seconds())
port = pd.to_numeric(df[PORT], errors="coerce")
rng = np.random.default_rng(0)
mismatches = 0
for i in rng.choice(len(df), 100, replace=False):
    same = (df["source_file"].to_numpy() == df["source_file"].iat[i]) & (port.to_numpy() == port.iat[i])
    want = int((same & (sec.to_numpy() >= sec.iat[i] - WINDOW_SECONDS) & (sec.to_numpy() <= sec.iat[i] - 1)).sum())
    mismatches += want != df["dst_port_flows_10s"].iat[i]
print("brute-force mismatches in 100 random rows:", mismatches)
print("rows without a value (no readable time or port):", int(df["dst_port_flows_10s"].isna().sum()))

brute-force mismatches in 100 random rows: 0
rows without a value (no readable time or port): 0


## 5. Clean (as v2; the duplicate check also ignores the port and feature 7, so the row count is unchanged)

In [6]:
n0 = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=BASE_FEATURES + COUNT_COLS)
# duplicates are judged on the original columns only, so row counts match earlier runs
df = df.drop_duplicates(subset=[c for c in df.columns
                                if c not in EXTRA_COLS + ["source_file", PORT, "dst_port_flows_10s"]])
print(f"rows: {n0:,} -> {len(df):,}")
print(df["tool"].value_counts())

rows: 8,997,323 -> 8,788,935
tool
Benign       7542570
HOIC          668461
LOIC-HTTP     576174
LOIC-UDP        1730
Name: count, dtype: int64


## 6. Build the features (as v2) and the feature sets compared here

In [7]:
df["fwd_bwd_ratio"] = df["Tot Fwd Pkts"] / (df["Tot Bwd Pkts"] + 1)                     # feature 3
df["no_reply"] = (df["Tot Bwd Pkts"] == 0).astype(int)                                  # feature 10
df["empty_payload"] = (df["Fwd Pkt Len Mean"] == 0).astype(int)                         # feature 10
df["iat_cv"] = (df["Flow IAT Std"] / (df["Flow IAT Mean"] + 1)).replace([np.inf, -np.inf], np.nan)  # feature 11
df["bwd_fwd_bytes_ratio"] = df["TotLen Bwd Pkts"] / (df["TotLen Fwd Pkts"] + 1)        # feature 12
df["data_share"] = (df["Fwd Act Data Pkts"] / df["Tot Fwd Pkts"]).replace([np.inf, -np.inf], np.nan)  # feature 15

GROUPS = {   # Week 3 numbering
    1: ("rate", ["Flow Pkts/s", "Flow Byts/s"]), 2: ("duration", ["Flow Duration"]),
    3: ("fwd/bwd ratio", ["fwd_bwd_ratio"]), 4: ("inter-arrival time", ["Flow IAT Mean", "Flow IAT Std"]),
    5: ("SYN flag count", ["SYN Flag Cnt"]), 6: ("forward packet length", ["Fwd Pkt Len Mean", "Fwd Pkt Len Std"]),
    7: ("port window", ["dst_port_flows_10s"]), 8: ("reply behaviour", ["Bwd Pkt Len Mean"]),
    9: ("connection-ending flags", ["RST Flag Cnt", "FIN Flag Cnt"]), 10: ("indicators", ["no_reply", "empty_payload"]),
    11: ("timing regularity", ["iat_cv"]), 12: ("response size ratio", ["bwd_fwd_bytes_ratio"]),
    13: ("sender timing", ["Fwd IAT Mean", "Fwd IAT Min"]), 14: ("activity pattern", ["Active Mean", "Idle Mean"]),
    15: ("payload signals", ["PSH Flag Cnt", "data_share"]),
}
def cols(groups):
    return [c for g in groups for c in GROUPS[g][1]]

F9 = cols((1, 2, 4, 5, 6, 3))         # the original nine columns (v2's FEATURES_BASE9, same order)
NO_SYN = cols((1, 2, 4, 6, 3))        # the same without SYN Flag Cnt (eight columns)
SETS = {"f9 (with SYN)": F9, "no SYN (8 columns)": NO_SYN}
for g in range(7, 16):
    SETS[f"SYN -> F{g} {GROUPS[g][0]}"] = NO_SYN + GROUPS[g][1]
F9P = SETS["SYN -> F7 port window"]   # the desktop app's default set f9p (same columns, same order)
print({name: len(c) for name, c in SETS.items()})
print("f9p:", F9P)

{'f9 (with SYN)': 9, 'no SYN (8 columns)': 8, 'SYN -> F7 port window': 9, 'SYN -> F8 reply behaviour': 9, 'SYN -> F9 connection-ending flags': 10, 'SYN -> F10 indicators': 10, 'SYN -> F11 timing regularity': 9, 'SYN -> F12 response size ratio': 9, 'SYN -> F13 sender timing': 10, 'SYN -> F14 activity pattern': 10, 'SYN -> F15 payload signals': 10}
f9p: ['Flow Pkts/s', 'Flow Byts/s', 'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'fwd_bwd_ratio', 'dst_port_flows_10s']


## 7. Splits (the same as the desktop app's training script)

- **all tools** (random): benign capped at 1,000,000 plus every attack flow, random 70/30 split stratified
  by tool. Optimistic: near-identical flows from the same burst land in both halves, and for feature 7 the
  training and test flows share the same seconds.
- **held out** (one tool never seen in training), benign split:
  `per_day` = the earlier 70% of each file's benign flows train, the later 30% test (v2 section 7);
  `by_file` = the earlier file's benign flows train, the later file's test (v2 section 7);
  `attack_period` = benign flows inside the held-out tool's attack window test, all others train.

In [8]:
def split(setup, benign_mode, seed):
    if setup == "all tools":
        ben = df[df["tool"] == "Benign"]
        d = pd.concat([ben.sample(min(BENIGN_CAP, len(ben)), random_state=seed), df[df["tool"] != "Benign"]])
        return train_test_split(d, test_size=1 - TRAIN_FRAC, stratify=d["tool"], random_state=seed)
    d = df[df[TIME].notna()]
    ben, att = d[d["tool"] == "Benign"], d[d["tool"] != "Benign"]
    held = att[att["tool"] == setup]
    if benign_mode == "per_day":
        in_test = ben.groupby("source_file")[TIME].rank(pct=True, method="first") > TRAIN_FRAC
    elif benign_mode == "by_file":
        order = ben.groupby("source_file")[TIME].min().sort_values().index
        in_test = ben["source_file"] == order[-1]
    else:  # attack_period
        in_test = (ben["source_file"].isin(held["source_file"].unique())
                   & ben[TIME].between(held[TIME].min(), held[TIME].max()))
    tr = pd.concat([ben[~in_test].sample(min(int((~in_test).sum()), BENIGN_CAP), random_state=seed),
                    att[att["tool"] != setup]])
    te = pd.concat([ben[in_test], held])
    return tr, te

## 8. Models

- **XGBoost**: the project's model (boosted trees), settings as in the app.
- **Random Forest**: 100 trees, maximum depth 20, at least 2 flows per leaf, `class_weight="balanced"`
  (the counterpart of XGBoost's `scale_pos_weight`). Bagged trees: each tree is trained on a random
  sample and the trees vote.
- **Logistic regression**: a weighted sum of the features. Rates and counts span many orders of
  magnitude, so the columns are log-scaled (keeping the sign) and standardised first;
  `class_weight="balanced"`.

None of the settings was tuned.

In [9]:
def xgboost(spw, seed):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
                         scale_pos_weight=spw, n_jobs=-1, random_state=seed)

def random_forest(spw, seed):
    return RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                  class_weight="balanced", n_jobs=-1, random_state=seed)

def signed_log(X):
    X = np.asarray(X, dtype=float)
    return np.sign(X) * np.log1p(np.abs(X))

def logistic(spw, seed):
    return make_pipeline(FunctionTransformer(signed_log), StandardScaler(),
                         LogisticRegression(class_weight="balanced", max_iter=2000))

MODELS = {"XGBoost": xgboost, "Random Forest": random_forest, "Logistic regression": logistic}

def evaluate(model, te, features):
    p = model.predict_proba(te[features])[:, 1]
    y, tool, pred = te["y"].to_numpy(), te["tool"].to_numpy(), p >= 0.5
    out = {"fpr": float(pred[y == 0].mean()), "false_alarms": int(pred[y == 0].sum()),
           "pr_auc": float(average_precision_score(y, p))}
    for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
        if (tool == t).any():
            out[f"recall {t}"] = float(pred[tool == t].mean())
    return out

## 9. The alert threshold, measured on the app's models

A flow raises an alert when its model score is at or above the threshold (default 0.5). Below are the app's
trained models (seed 0) on their own test splits, which are rebuilt here exactly as the training script built
them. The first check is that recall at 0.5 equals the number stored with each model.

Choosing a threshold on the same data the result is reported on is optimistic. The group decided to choose it
after testing, so the table is the evidence, read with that caveat.

In [10]:
import json
MODELS_DIR = Path("../DDoSDetector/models")

def app_model(name):
    meta = json.loads((MODELS_DIR / f"{name}.meta.json").read_text(encoding="utf-8"))
    model = XGBClassifier()
    model.load_model(MODELS_DIR / f"{name}.json")
    assert meta["feature_columns"] == F9P, "the app model does not use the f9p columns"
    return model, meta

THRESHOLDS = [0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.01, 0.001, 0.0001]
CASES = {"all tools (random split)": ("all_tools_f9p", "all tools", "random"),
         "HOIC held out (benign split by day)": ("hoic_held_out_f9p", "HOIC", "per_day"),
         "LOIC-HTTP held out (benign split by day)": ("loic_http_held_out_f9p", "LOIC-HTTP", "per_day")}
APP = {}
rows = []
for label, (name, setup, mode) in CASES.items():
    model, meta = app_model(name)
    tr, te = split(setup, mode, 0)
    p = model.predict_proba(te[F9P])[:, 1]
    APP[label] = (model, te, p)
    y, tool = te["y"].to_numpy(), te["tool"].to_numpy()
    at05 = {t: round(float((p[tool == t] >= 0.5).mean()), 4) for t in meta["metrics"]["recall"]}
    print(f"{name}: recall at 0.5 here {at05} | stored with the model {meta['metrics']['recall']}")
    for thr in THRESHOLDS:
        pred = p >= thr
        row = {"case": label, "threshold": thr, "false alarms": int(pred[y == 0].sum()),
               "false positive rate": round(float(pred[y == 0].mean()), 5)}
        for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
            if (tool == t).any():
                row[f"recall {t}"] = round(float(pred[tool == t].mean()), 4)
        rows.append(row)
thr_table = pd.DataFrame(rows)
for label, g in thr_table.groupby("case", sort=False):
    print(f"\n=== {label} ({int((APP[label][1]['y'] == 0).sum()):,} benign test flows)")
    print(g.drop(columns="case").dropna(axis=1, how="all").to_string(index=False))

all_tools_f9p: recall at 0.5 here {'HOIC': 1.0, 'LOIC-HTTP': 0.9998, 'LOIC-UDP': 1.0} | stored with the model {'HOIC': 1.0, 'LOIC-HTTP': 0.9998, 'LOIC-UDP': 1.0}


hoic_held_out_f9p: recall at 0.5 here {'HOIC': 0.3967} | stored with the model {'HOIC': 0.3967}


loic_http_held_out_f9p: recall at 0.5 here {'LOIC-HTTP': 0.0023} | stored with the model {'LOIC-HTTP': 0.0023}



=== all tools (random split) (300,000 benign test flows)
 threshold  false alarms  false positive rate  recall HOIC  recall LOIC-HTTP  recall LOIC-UDP
    0.9000            50              0.00017       0.9999            0.9987           0.9981
    0.7000            84              0.00028       0.9999            0.9997           1.0000
    0.5000           103              0.00034       1.0000            0.9998           1.0000
    0.3000           117              0.00039       1.0000            0.9999           1.0000
    0.2000           130              0.00043       1.0000            1.0000           1.0000
    0.1000           163              0.00054       1.0000            1.0000           1.0000
    0.0500           187              0.00062       1.0000            1.0000           1.0000
    0.0100           358              0.00119       1.0000            1.0000           1.0000
    0.0010           996              0.00332       1.0000            1.0000           1.0000
  

In [11]:
# How low do the scores of the attack flows that the held-out models miss go?
for label in list(CASES)[1:]:
    model, te, p = APP[label]
    tool = label.split(" held out")[0]
    s = p[te["tool"].to_numpy() == tool]
    b = p[te["tool"].to_numpy() == "Benign"]
    q = np.quantile(s, [0.1, 0.25, 0.5, 0.6])
    print(f"{label}: {tool} score quantiles 10/25/50/60%: " + ", ".join(f"{v:.2e}" for v in q)
          + f" | benign median {np.median(b):.2e}")

HOIC held out (benign split by day): HOIC score quantiles 10/25/50/60%: 1.52e-05, 9.08e-04, 1.36e-03, 3.28e-03 | benign median 3.03e-07


LOIC-HTTP held out (benign split by day): LOIC-HTTP score quantiles 10/25/50/60%: 1.48e-05, 4.49e-05, 5.07e-05, 4.51e-04 | benign median 8.95e-08


## 10. Top features (SHAP)

Mean absolute SHAP value of each input (how much it moves the score, in log-odds) for 2,000 test flows of
the all-tools model, and, for the HOIC-held-out model, the average SHAP value of each input for the HOIC flows
it catches and the ones it misses (positive pushes towards DDoS, negative towards benign).

In [12]:
import shap

def mean_abs_shap(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    return pd.Series(np.abs(sv).mean(axis=0), index=X.columns).sort_values(ascending=False)

model, te, p = APP["all tools (random split)"]
sample = te.sample(2000, random_state=0)
print("All-tools model, mean |SHAP| on 2,000 test flows:")
print(mean_abs_shap(model, sample[F9P]).round(3).to_string())

model, te, p = APP["HOIC held out (benign split by day)"]
hoic = te[te["tool"] == "HOIC"].assign(score=p[(te["tool"] == "HOIC").to_numpy()])
caught = hoic[hoic["score"] >= 0.5].sample(1000, random_state=0)
missed = hoic[hoic["score"] < 0.5].sample(1000, random_state=0)
explainer = shap.TreeExplainer(model)
side = pd.DataFrame({"caught HOIC flows": explainer.shap_values(caught[F9P]).mean(axis=0),
                     "missed HOIC flows": explainer.shap_values(missed[F9P]).mean(axis=0)}, index=F9P)
print("\nHOIC-held-out model, average SHAP value per input (1,000 caught and 1,000 missed HOIC flows):")
print(side.round(2).sort_values("missed HOIC flows").to_string())
print("\nMedian input values, caught vs missed:")
print(pd.DataFrame({"caught": caught[F9P].median(), "missed": missed[F9P].median()}).to_string())

C:\Users\Xavie\source\repos\DDOS-ML-Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All-tools model, mean |SHAP| on 2,000 test flows:


dst_port_flows_10s    8.086
Fwd Pkt Len Mean      1.873
fwd_bwd_ratio         1.661
Flow Duration         1.183
Flow IAT Std          0.852
Fwd Pkt Len Std       0.525
Flow IAT Mean         0.447
Flow Byts/s           0.441
Flow Pkts/s           0.430



HOIC-held-out model, average SHAP value per input (1,000 caught and 1,000 missed HOIC flows):
                    caught HOIC flows  missed HOIC flows
Flow IAT Mean                   -1.83              -7.99
Fwd Pkt Len Mean                 0.64              -2.85
Flow Duration                   -2.25              -2.81
Flow IAT Std                     0.08              -0.05
fwd_bwd_ratio                    0.22              -0.01
Fwd Pkt Len Std                 -0.03               0.07
Flow Byts/s                     -0.06               0.15
Flow Pkts/s                      0.72               0.68
dst_port_flows_10s               6.84               4.44

Median input values, caught vs missed:
                          caught       missed
Flow Pkts/s           138.893917   951.669351
Flow Byts/s             0.000000     0.000000
Flow Duration       14399.500000  2535.500000
Flow IAT Mean       14399.500000  1903.166667
Flow IAT Std            0.000000     0.000000
Fwd Pkt Len Mean   

## 11. Retraining demonstration: a new attack tool appears

**Scenario.** The model was trained without one attack tool (the app's held-out setup). That tool now attacks.
Analysts confirm the incident early and label the **first flows of the attack** (the first 100, 1,000 or
10,000 by start time) as attacks. The labels are added to the training data with the training script's
`--extra-labelled` mechanism (here: the same rows added in code), and XGBoost is retrained. The model is then
tested on the **rest of the attack** (every flow of that tool after its first 10,000, the same for every run)
and on the same benign test flows.

Two ways of labelling are compared:

- **whole incident**: analysts label the first flows of the attack, flagged or not (they know the incident
  and its time window);
- **alerts only**: analysts label only the flows the old model flagged among those first 10,000 flows, which
  is what alert-by-alert review gives (so fewer labels may exist than asked for).

A third option is a **quick update**: instead of retraining from scratch, 50 trees are added to the existing
model, trained on 1,000 whole-incident flows plus 100,000 rows of the original training data (so it does not
forget the tools it already knew). XGBoost can continue training an existing model in this way.

Seeds 0 and 1. Same settings as the app (300 trees, depth 6, learning rate 0.1, class weights).

In [13]:
def fit(train, seed, base=None, n_estimators=300):
    spw = (train["y"] == 0).sum() / (train["y"] == 1).sum()
    m = XGBClassifier(n_estimators=n_estimators, max_depth=6, learning_rate=0.1, tree_method="hist",
                      scale_pos_weight=spw, n_jobs=-1, random_state=seed)
    t1 = time.perf_counter()
    m.fit(train[F9P], train["y"], xgb_model=None if base is None else base.get_booster())
    return m, time.perf_counter() - t1

def score_on(model, te_ben, te_att):
    pa = model.predict_proba(te_att[F9P])[:, 1] >= 0.5
    pb = model.predict_proba(te_ben[F9P])[:, 1] >= 0.5
    return {"recall on rest of attack": round(float(pa.mean()), 4),
            "false alarms": int(pb.sum()), "false positive rate": round(float(pb.mean()), 5)}

demo = []
for tool in ["HOIC", "LOIC-HTTP"]:
    for seed in SEEDS:
        tr, te = split(tool, "per_day", seed)
        att = te[te["tool"] == tool].sort_values(TIME, kind="stable")
        pool, rest = att.iloc[:10_000], att.iloc[10_000:]
        te_ben = te[te["tool"] == "Benign"]
        base, secs = fit(tr, seed)
        flagged = pool[base.predict_proba(pool[F9P])[:, 1] >= 0.5]
        info = {"tool": tool, "seed": seed, "of first 10,000 flagged": len(flagged), "rest of attack": len(rest)}
        demo.append({**info, "model": "before (no labels)", "labelled flows": 0, "train s": round(secs, 1),
                     **score_on(base, te_ben, rest)})
        for source, frame in [("whole incident", pool), ("alerts only", flagged)]:
            used = 0
            for n in [100, 1000, 10000]:
                fb = frame.iloc[:n]                  # the first n by start time
                if len(fb) == used:                  # no more labels available: same run as before
                    continue
                used = len(fb)
                m, secs = fit(pd.concat([tr, fb]), seed)
                demo.append({**info, "model": f"retrained, {source}", "labelled flows": len(fb),
                             "train s": round(secs, 1), **score_on(m, te_ben, rest)})
        fb = pool.iloc[:1000]
        upd = pd.concat([tr.sample(100_000, random_state=seed), fb])
        m, secs = fit(upd, seed, base=base, n_estimators=50)
        demo.append({**info, "model": "quick update (+50 trees), whole incident", "labelled flows": len(fb),
                     "train s": round(secs, 1), **score_on(m, te_ben, rest)})
        span = pool[TIME].max() - pool[TIME].min()
        print(f"{tool} seed {seed} done: the first 10,000 flows span {span}, {len(flagged):,} of them flagged; "
              f"{len(rest):,} flows left to test on")
demo = pd.DataFrame(demo)
cols = ["tool", "seed", "model", "labelled flows", "recall on rest of attack", "false alarms",
        "false positive rate", "train s"]
print(demo[cols].to_string(index=False))

HOIC seed 0 done: the first 10,000 flows span 0 days 00:00:17, 3,986 of them flagged; 658,461 flows left to test on


HOIC seed 1 done: the first 10,000 flows span 0 days 00:00:17, 3,961 of them flagged; 658,461 flows left to test on


LOIC-HTTP seed 0 done: the first 10,000 flows span 0 days 09:00:43, 718 of them flagged; 566,174 flows left to test on


LOIC-HTTP seed 1 done: the first 10,000 flows span 0 days 09:00:43, 719 of them flagged; 566,174 flows left to test on
     tool  seed                                    model  labelled flows  recall on rest of attack  false alarms  false positive rate  train s
     HOIC     0                       before (no labels)               0                    0.3966          4514              0.00199      7.6
     HOIC     0                retrained, whole incident             100                    0.3966          4557              0.00201      7.9
     HOIC     0                retrained, whole incident            1000                    0.3989          4647              0.00205      7.8
     HOIC     0                retrained, whole incident           10000                    0.9928          5468              0.00242      7.7
     HOIC     0                   retrained, alerts only             100                    0.3966          4509              0.00199      7.7
     HOIC     0        

## 12. Findings

4 November 2026, from the outputs above. The app's models are seed 0; the retraining runs use seeds 0 and 1.

**Check.** Recall at 0.5 recomputed here equals the value stored with each of the app's models, so the splits are
the same as the training script's.

**9. Threshold**
- For the tools the model knows (all-tools model, random split), every threshold from 0.1 to 0.7 gives recall of
  0.9997 or more for each tool, with 84 to 163 false alarms among 300,000 benign test flows (103 at 0.5).
- For tools it has not seen, recall does not move between 0.7 and 0.01: HOIC 0.397, LOIC-HTTP 0.002 to 0.003. The
  missed flows score far below 0.01 (median 0.0014 for HOIC and 0.00005 for LOIC-HTTP). Only at 0.001 does recall
  rise (HOIC 0.562, LOIC-HTTP 0.223), with 7 to 10 times as many false alarms (30,854 and 12,278 against 4,514 and
  1,218 at 0.5).
- So we propose keeping 0.5. A lower threshold does not fix a new attack tool. These numbers are measured on the
  same test data, so they are optimistic.

**10. Top features**
- All-tools model: feature 7 (`dst_port_flows_10s`) moves the score most (mean |SHAP| 8.1 in log-odds), then
  forward packet length (1.9), the forward to backward ratio (1.7) and flow duration (1.2).
- HOIC-held-out model: the HOIC flows it misses are pushed towards benign mainly by the inter-arrival time (average
  SHAP −8.0, against −1.8 for the HOIC flows it catches) and the forward packet length (−2.9 against +0.6). The
  missed flows are shorter and faster (median duration about 2,500 µs against 14,400) while their port count is
  just as high (about 4,000), which pushes them towards DDoS but not enough.

**11. Retraining**
- With the first 10,000 flows of the new attack labelled (17 seconds of HOIC), retraining raised recall on the
  rest of the attack from 0.395–0.397 to 0.993–0.998 for HOIC and from 0.001 to 0.996–0.998 for LOIC-HTTP. Each
  retraining took 8 to 11 seconds. False alarms rose from 4,282–4,514 to 5,105–5,468 (HOIC) and from 966–1,218 to
  7,951–8,320 (LOIC-HTTP, 0.35–0.37% of benign flows), so a retrained model has to be checked before it replaces
  the old one.
- 100 or 1,000 flows from the very start of the attack changed little (HOIC stayed at about 0.40; LOIC-HTTP
  reached 0.087–0.088 with 1,000). The first 10,000 LOIC-HTTP flows span about nine hours, because a few flows
  labelled LOIC-HTTP appear long before the main attack.
- Labelling only the alerts (every flagged flow among the first 10,000: about 3,970 for HOIC and 720 for LOIC-HTTP)
  left recall at 0.418–0.424 (HOIC) and 0.001 (LOIC-HTTP), because the model already catches those flows.
  Feedback should therefore cover the whole incident, not only the alerts.
- Adding 50 trees to the old model (0.2 to 0.3 seconds) was unreliable: for HOIC it stopped flagging anything
  (seed 0) or raised 50,016 false alarms (seed 1). A full retrain is fast enough, so we retrain from scratch.

**Caveats.** Two seeds; settings not tuned; the held-out tests use the 21 February benign flows, most of which look
like the server's side of the HOIC connections (see `ddos_update4_experiments.ipynb`, section 7).